# 03b · Pachetul de validare pentru irigare (80 de parcele de primăvară, D26 pct. 8)
Selectează 80 de parcele cu cultură de primăvară în 2024 (40 `irigat` + 40 `neirigat` după regula S1 + pivoți), răspândite pe blocuri de 5 km pe tot AOI-ul:

- `data/out/irrigation_labels.gpkg`: cele 80 de parcele pentru verificare în QGIS (coloana `irigat` e goală).
- `data/interim/irrigation_label_hints.parquet`: clasa atribuită de regulă (`irigat` / `neirigat`), păstrată separat pentru evaluarea oarbă.
- `scripts/label_sheets.py` generează foile de contact `data/interim/label_sheets/sheet_01.png` … `sheet_04.png` (`iul NIR / aug NIR / aug`) și `data/out/labels.csv` (`id,irigat`).

In [1]:
import os, sys
if not os.path.exists('gisday_config.py') and os.path.exists('../gisday_config.py'):
    os.chdir('..')
sys.path.insert(0, os.getcwd())
from gisday_config import *
from gisday_lib import select_validation_80
import numpy as np, pandas as pd, geopandas as gpd

parcels = gpd.read_parquet(OUT / 'parcels.parquet').to_crs(CRS)
irr = pd.read_parquet(OUT / 'irrigation_2024.parquet')
D = parcels[['parcel_id', 'geometry']].merge(irr, on='parcel_id')
c = D.geometry.centroid
D['block'] = (c.x // 5000).astype(int).astype(str) + '_' + (c.y // 5000).astype(int).astype(str)

# Selectăm dintre parcelele unde clasa coincide în ambele variante (Irigat_B ⊂ Irigat_A și Neirigat_A ⊂ Neirigat_B),
# astfel încât toate cele 80 de parcele (40 irigat + 40 neirigat) rămân 100% valide indiferent de decizia pe o7_0615_0627.
D['clasa_consens'] = np.where(
    D.clasa_irigare.eq('irigat') & D.clasa_irigare_fara_0627.eq('irigat'), 'irigat',
    np.where(D.clasa_irigare.eq('neirigat') & D.clasa_irigare_fara_0627.eq('neirigat'), 'neirigat', 'nedeterminat')
)
pick = select_validation_80(D, class_col='clasa_consens', n_per_class=40, min_area_ha=1.0, max_per_block=1, seed=2024)
L = D.merge(pick, on='parcel_id')
L[['parcel_id', 'clasa_regula', 'is_pivot', 'n_events_s1', 'n_events_s1_fara_0627', 'max_dvv_dry']].to_parquet(
    INTERIM / 'irrigation_label_hints.parquet'
)
L['irigat'] = pd.array([pd.NA] * len(L), dtype='Int32')
LABELS = OUT / 'irrigation_labels.gpkg'
gpd.GeoDataFrame(L[['parcel_id', 'irigat', 'area_ha', 'geometry']], geometry='geometry', crs=CRS).to_file(
    LABELS, layer='etichete', driver='GPKG'
)
print(f'{len(L)} parcele de validare salvate în {LABELS.name}:', L.clasa_regula.value_counts().to_dict(), '| blocuri 5 km acoperite:', L.block.nunique())

80 parcele de validare salvate în irrigation_labels.gpkg: {'irigat': 40, 'neirigat': 40} | blocuri 5 km acoperite: 55
